# Endoscopy RRDB/Real-ESRGAN Transfer-Learning Training

Separate, opt-in experiment: fine-tunes a **pretrained Real-ESRGAN generator**
(RRDBNet, dense residual blocks, no BatchNorm) on the endoscopy dataset,
instead of training the paper-faithful SRGAN generator from scratch. Uses its
own files (`rrdb_generator.py`, `train_rrdb.py`) and its own checkpoint
dir/weights (`srgan_v_transfer.pth`) — none of this touches the paper-faithful
pipeline or its checkpoints.

**Known issue + fix applied (not yet validated on a real run):** earlier
training showed the discriminator dominating (D accuracy pinned near 100%
early), traced to a 10x learning-rate gap between G and D on the same cosine
decay schedule. `train_rrdb.py` now defaults `--lr_d` to `3e-5` (~3x `lr_g`
instead of ~10x) and logs D's real/fake classification accuracy every epoch —
watch `D_real_acc`/`D_fake_acc` in the training cell below. If they stay
pinned near 1.0 for many epochs, the fix didn't work and it needs a stronger
intervention (D:G update ratio, label smoothing) — see the training-track
slides in the deck for the full diagnosis.

Mirrors the clean HSV/YIQ notebooks' structure (local-disk data, Drive backup only).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Kaggle authentication

**Only needed the very first time ever** — skip if a Drive backup already exists (cell 4).

In [ ]:
import os

KAGGLE_TOKEN = 'PASTE_YOUR_TOKEN_HERE'

if KAGGLE_TOKEN != 'PASTE_YOUR_TOKEN_HERE':
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/access_token', 'w') as f:
        f.write(KAGGLE_TOKEN)
    os.system('chmod 600 /root/.kaggle/access_token')
    print('Kaggle token saved.')
else:
    print('Skipped -- fine if a Drive backup already exists (cell 4 will use it).')

## 3. Install dependencies

In [ ]:
!pip install -q kaggle opencv-python-headless torch torchvision scikit-image lpips

## 4. Get the dataset — local disk first, Drive as backup only

Same dataset as the HSV/YIQ pipelines (RRDB just swaps the generator, not the
data) — if you already built a Drive backup from an earlier notebook, this
cell reuses it directly, no re-download needed.

In [ ]:
import os

LOCAL_DATA = '/content/data'
DRIVE_BACKUP = '/content/drive/MyDrive/endoscopy_srgan/data_backup'
DATASETS = ['kvasir', 'cvc_clinicdb', 'etis_larib']

def _has_all(root):
    return all(os.path.isdir(f'{root}/{d}') for d in DATASETS)

if _has_all(LOCAL_DATA):
    print('Local data already present this session -- skipping.')
elif _has_all(DRIVE_BACKUP):
    print('Restoring from Drive backup (fast local copy)...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    for d in DATASETS:
        os.system(f'cp -r {DRIVE_BACKUP}/{d} {LOCAL_DATA}/')
    print('Restored from Drive backup.')
else:
    print('No local data or Drive backup found -- downloading fresh from Kaggle to local disk...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    os.system(f'kaggle datasets download -d meetnagadia/kvasir-dataset -p {LOCAL_DATA}/kvasir --unzip')
    os.system(f'kaggle datasets download -d balraj98/cvcclinicdb -p {LOCAL_DATA}/cvc_clinicdb --unzip')
    os.system(f'kaggle datasets download -d nguyenvoquocduong/etis-laribpolypdb -p {LOCAL_DATA}/etis_larib --unzip')
    print('Downloaded. Backing up to Drive for future sessions...')
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    for d in DATASETS:
        os.system(f'cp -r {LOCAL_DATA}/{d} {DRIVE_BACKUP}/')
    print('Backed up to Drive.')

## 5. Build the manifest (local paths)

In [ ]:
import glob, random, json

paths = []
paths += glob.glob(f'{LOCAL_DATA}/kvasir/kvasir-dataset/*/*.jpg')
paths += glob.glob(f'{LOCAL_DATA}/cvc_clinicdb/PNG/Original/*.png')
paths += glob.glob(f'{LOCAL_DATA}/etis_larib/images/*.png')
print('total images found:', len(paths))

random.seed(42)
random.shuffle(paths)
n_val = int(0.1 * len(paths))
manifest = {'train': paths[n_val:], 'val': paths[:n_val]}

MANIFEST_PATH = f'{LOCAL_DATA}/manifest.json'
with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f)
print('train:', len(manifest['train']), '| val:', len(manifest['val']))

## 6. Clone or update the repo

In [ ]:
import os

if os.path.exists('/content/repo/.git'):
    %cd /content/repo
    !git pull
else:
    !git clone https://github.com/knah1d/unsharp-image_processing.git /content/repo
    %cd /content/repo

## 7. Download the pretrained Real-ESRGAN base weights

One-time download -- only needed on a genuinely fresh run (a resumed run
reuses the fine-tuned checkpoint instead, see `train_rrdb.py`'s docstring).
`x2plus` is chosen specifically because it's a native 2x model, matching this
pipeline's 2x scale.

In [ ]:
!wget -q -nc https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth
print('done')

## 8. Smoke test (recommended before a long run)

Cheap sanity check on the RRDB fine-tuning path specifically -- catches bugs
in a couple of minutes and confirms `D_real_acc`/`D_fake_acc` are printing.

In [ ]:
!python train_rrdb.py \
    --manifest /content/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints_rrdb_smoketest \
    --pretrained RealESRGAN_x2plus.pth \
    --epochs 3 --pretrain_epochs 1 --batch_size 8

## 9. Real training run

Checkpoints go to an RRDB-specific Drive folder (`checkpoints_rrdb`), fully
separate from the HSV and YIQ pipelines. Auto-resumes via `rrdb_last.pth` if
this cell is rerun after a disconnect.

**Watch `D_real_acc` / `D_fake_acc` in the printed output each epoch** -- this
is the actual test of the discriminator-dominance fix. Healthy training keeps
both roughly in a 0.6-0.8 range; if they pin near 1.0 early and stay there,
the fix didn't hold and needs a stronger intervention before trusting this
path further.

In [ ]:
!python train_rrdb.py \
    --manifest /content/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints_rrdb \
    --pretrained RealESRGAN_x2plus.pth \
    --epochs 30 --pretrain_epochs 2 --batch_size 8

## 10. (Optional) Explicit LR override

Only needed if the default `--lr_d 3e-5` still shows D dominating in the
logs above -- try narrowing the gap further (e.g. `--lr_d 2e-5`) rather than
widening it back out.

In [ ]:
!python train_rrdb.py \
    --manifest /content/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints_rrdb \
    --pretrained RealESRGAN_x2plus.pth \
    --epochs 30 --pretrain_epochs 2 --batch_size 8 --lr_d 2e-5